# MultiFold

This notebook contains the implementation of the modern, deep-learning-based unfolding technique called **MultiFold**. 

Firstly, we require several libraries:
* os to create folders and extract OS variables;
* ROOT to extract data from the `.root` files;
* NumPy to store and process the data;
* Matplotlib for plotting;
* OmniFold from the `omnifold.py` file for unfolding,
* Keras classes for construction of the neural network.

In [ ]:
import os
import argparse
import sys

# extract the active Conda environment path
conda_env_path = os.environ.get("CONDA_PREFIX", "")

# tell TensorFlow's XLA compiler where Conda put C++ libraries (COMMENT if you're not using an NVIDIA GPU and did not run this command: conda install -c nvidia cuda-nvcc)
os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={conda_env_path}"

import gc
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import omnifold as of

from keras.layers import Dense, Input
from keras.models import Model
from sklearn.preprocessing import StandardScaler

Similarly to its predecessors, this notebook can also be converted to a script via `jupyter nbconvert --to script multifold-ea.ipynb`. The script then takes seed arguments, e.g., `python3 multifold-ea.py --seedTrainingBiased 2 --seedTrainingMB 20000 --seedUnfolding 30000`. Additionally, it can be submitted to the FNSPE Sunrise cluster – recommended submission command: `qsub  -m e -M youremail@institution.com -l select=1:ncpus=10:mem=250gb:host=sunset30 -q normal multifold_pp510.sh`.

In [ ]:
# set values when running as a Jupyter notebook
if 'ipykernel' in sys.modules:
    print("Jupyter environment detected! Assuming that MultiFold is ran locally as a Jupyter notebook.")

    # set the default seeds for neural network training and for the dataset to be unfolded
    seedTrainingBiased = 2
    seedTrainingMB = 20000
    seedUnfolding = 22

# otherwise, python script is assumed and the seed is loaded from the command line
else:

    # initialize the parser which loads the seed argument
    parser = argparse.ArgumentParser()

    # define the arguments to load
    parser.add_argument('--seedTrainingBiased', type=int, required=True)
    parser.add_argument('--seedTrainingMB', type=int, required=True)
    parser.add_argument('--seedUnfolding', type=int, required=True)

    # parse the argument from the command line
    args = parser.parse_args()

    # set the seed
    seedTrainingBiased = args.seedTrainingBiased
    seedTrainingMB = args.seedTrainingMB
    seedUnfolding = args.seedUnfolding

    # restrict the number of cores used by tensorflow for single mathematical operation to 8 cores -- optimal for small neural networks used in this notebook (larger number of cores would cause a bottleneck as the CPU would spend most of the time distributing small tasks to many cores and then combining the results)
    tf.config.threading.set_intra_op_parallelism_threads(8)

    # restrict the number of cores used by tensorflow for independent mathematical operations to 2 cores -- the fully connected Dense neural networks created below are strictly sequential, thus there are zero independent operations --> we prevent tensorflow from looking for independent operations
    tf.config.threading.set_inter_op_parallelism_threads(2)

We load columns of our observables for training and unfolding from the respective `.root` files into NumPy arrays utilizing RDataFrame.

We will train the neural network on a small minimum bias dataset enriched with events that contain at least one 20 GeV parton-parton scattering (2 million events in total for training). The biased dataset has to be generated using the `pythia8_generate-tree.ipynb` notebook with modified PYTHIA8 configuration in `cpp/generate_events.cpp`. Unfolding will be carried out on the minimum bias dataset.

Firstly, we will unfold only the multiplicity distribution using UniFold. Then, we will unfold the spherocity observables utilizing MultiFold.

In [ ]:
# connect RDataFrame to the training and unfolding TTrees with observables and convert the values in the trees to 32-bit floats, saving 50 % of RAM as they would be loaded as 64-bit floats
dfTrainingNchBiased = (
    ROOT.RDataFrame("multiplicity", f"data/{seedTrainingBiased}/observables{seedTrainingBiased}Nch.root")
        .Range(500000) # load only first 500 000 events
        .Redefine("NchTrue", "(float)NchTrue")
        .Redefine("NchSmeared", "(float)NchSmeared")
)
dfTrainingNchMB = (
    ROOT.RDataFrame("multiplicity", f"data/{seedTrainingMB}/observables{seedTrainingMB}Nch.root")
        .Range(1500000) # load only first 1 500 000 events
        .Redefine("NchTrue", "(float)NchTrue")
        .Redefine("NchSmeared", "(float)NchSmeared")
)
dfUnfoldingNch = (
    ROOT.RDataFrame("multiplicity", f"data/{seedUnfolding}/observables{seedUnfolding}Nch.root")
        .Redefine("NchTrue", "(float)NchTrue")
        .Redefine("NchSmeared", "(float)NchSmeared")
)

dfTrainingNchCutBiased = (
    ROOT.RDataFrame("observables_NchCut", f"data/{seedTrainingBiased}/observables{seedTrainingBiased}NchCut.root")
        .Range(500000)
        .Redefine("NchTrue", "(float)NchTrue")
        .Redefine("S0True", "(float)S0True")
        .Redefine("S0pT1True", "(float)S0pT1True")
        .Redefine("NchSmeared", "(float)NchSmeared")
        .Redefine("S0Smeared", "(float)S0Smeared")
        .Redefine("S0pT1Smeared", "(float)S0pT1Smeared")
)
dfTrainingNchCutMB = (
    ROOT.RDataFrame("observables_NchCut", f"data/{seedTrainingMB}/observables{seedTrainingMB}NchCut.root")
        .Range(1500000)
        .Redefine("NchTrue", "(float)NchTrue")
        .Redefine("S0True", "(float)S0True")
        .Redefine("S0pT1True", "(float)S0pT1True")
        .Redefine("NchSmeared", "(float)NchSmeared")
        .Redefine("S0Smeared", "(float)S0Smeared")
        .Redefine("S0pT1Smeared", "(float)S0pT1Smeared")
)
dfUnfoldingNchCut = (
    ROOT.RDataFrame("observables_NchCut", f"data/{seedUnfolding}/observables{seedUnfolding}NchCut.root")
        .Redefine("NchTrue", "(float)NchTrue")
        .Redefine("S0True", "(float)S0True")
        .Redefine("S0pT1True", "(float)S0pT1True")
        .Redefine("NchSmeared", "(float)NchSmeared")
        .Redefine("S0Smeared", "(float)S0Smeared")
        .Redefine("S0pT1Smeared", "(float)S0pT1Smeared")
)

# load columns into NumPy arrays
dataTrainingNchBiased = dfTrainingNchBiased.AsNumpy()
dataTrainingNchMB = dfTrainingNchMB.AsNumpy()
dataUnfoldingNch = dfUnfoldingNch.AsNumpy()

dataTrainingNchCutBiased = dfTrainingNchCutBiased.AsNumpy()
dataTrainingNchCutMB = dfTrainingNchCutMB.AsNumpy()
dataUnfoldingNchCut = dfUnfoldingNchCut.AsNumpy()

# define arrays compatible with the neural network defined below (we must concatenate the biased and MB events)
dataTrainingNchTrueUnshuffled = np.column_stack((
    np.concatenate((dataTrainingNchBiased["NchTrue"], dataTrainingNchMB["NchTrue"])),
))
dataTrainingNchSmearedUnshuffled = np.column_stack((
    np.concatenate((dataTrainingNchBiased["NchSmeared"], dataTrainingNchMB["NchSmeared"])),
))
dataUnfoldingNchTrue = np.column_stack((
    dataUnfoldingNch["NchTrue"],
))
dataUnfoldingNchSmeared = np.column_stack((
    dataUnfoldingNch["NchSmeared"],
))

del dataTrainingNchBiased
del dataTrainingNchMB
del dataUnfoldingNch
gc.collect() # force python's garbage collector to cleanup the memory right now

dataTrainingNchCutTrueUnshuffled = np.column_stack((
    np.concatenate((dataTrainingNchCutBiased["NchTrue"], dataTrainingNchCutMB["NchTrue"])),
    np.concatenate((dataTrainingNchCutBiased["S0True"], dataTrainingNchCutMB["S0True"])),
    np.concatenate((dataTrainingNchCutBiased["S0pT1True"], dataTrainingNchCutMB["S0pT1True"]))
))
dataTrainingNchCutSmearedUnshuffled = np.column_stack((
    np.concatenate((dataTrainingNchCutBiased["NchSmeared"], dataTrainingNchCutMB["NchSmeared"])),
    np.concatenate((dataTrainingNchCutBiased["S0Smeared"], dataTrainingNchCutMB["S0Smeared"])),
    np.concatenate((dataTrainingNchCutBiased["S0pT1Smeared"], dataTrainingNchCutMB["S0pT1Smeared"]))
))
dataUnfoldingNchCutTrue = np.column_stack((
    dataUnfoldingNchCut["NchTrue"],
    dataUnfoldingNchCut["S0True"],
    dataUnfoldingNchCut["S0pT1True"]
))
dataUnfoldingNchCutSmeared = np.column_stack((
    dataUnfoldingNchCut["NchSmeared"],
    dataUnfoldingNchCut["S0Smeared"],
    dataUnfoldingNchCut["S0pT1Smeared"]
))

del dataTrainingNchCutBiased
del dataTrainingNchCutMB
del dataUnfoldingNchCut
gc.collect() # memory cleanup

# shuffle the training arrays
indicesNch = np.random.permutation(len(dataTrainingNchTrueUnshuffled))
dataTrainingNchTrue = dataTrainingNchTrueUnshuffled[indicesNch]
dataTrainingNchSmeared = dataTrainingNchSmearedUnshuffled[indicesNch]

indicesNchCut = np.random.permutation(len(dataTrainingNchCutTrueUnshuffled))
dataTrainingNchCutTrue = dataTrainingNchCutTrueUnshuffled[indicesNchCut]
dataTrainingNchCutSmeared = dataTrainingNchCutSmearedUnshuffled[indicesNchCut]

del dataTrainingNchTrueUnshuffled
del dataTrainingNchSmearedUnshuffled
del dataTrainingNchCutTrueUnshuffled
del dataTrainingNchCutSmearedUnshuffled
gc.collect() # memory cleanup

## Setup of the Neural Networks

We set up a classifier neural networks using the imported Keras classes:
- number of nodes per hidden layer: 20;
- activation function of hidden layers: relu = Rectified Linear Unit $f(x) = \max(0, x)$;
- activation function of the output layer: sigmoid ... ensures that the neural network is a classifier.

Firstly, we set up the UniFold neural network for multiplicity unfolding.

In [ ]:
# specify the number of inputs (observables) to the neural network
inputs = Input((1, ))

# define the hidden layers using the fully connected Dense layers
hiddenLayer1 = Dense(20, activation='relu')(inputs)
hiddenLayer2 = Dense(20, activation='relu')(hiddenLayer1)
hiddenLayer3 = Dense(20, activation='relu')(hiddenLayer2)

# define the outputs
outputs = Dense(1, activation='sigmoid')(hiddenLayer3)

# define the neural network object
UniFold = Model(inputs = inputs, outputs = outputs)

# print the summary of the neural network
UniFold.summary()

Secondly, we define the neural network for spherocities (MultiFold).

In [ ]:
# specify the number of inputs (observables) to the neural network
inputs = Input((3, ))

# define the hidden layers using the fully connected Dense layers
hiddenLayer1 = Dense(20, activation='relu')(inputs)
hiddenLayer2 = Dense(20, activation='relu')(hiddenLayer1)
hiddenLayer3 = Dense(20, activation='relu')(hiddenLayer2)

# define the outputs
outputs = Dense(1, activation='sigmoid')(hiddenLayer3)

# define the neural network object
MultiFold = Model(inputs = inputs, outputs = outputs)

# print the summary of the neural network
MultiFold.summary()

We must also scale the data for the neural networks. The data must have mean of 0 and standard deviation of 1. We will use the StandardScaler class from scikit-learn for this.

In [ ]:
# initialize the scalers
scalerNchTrue = StandardScaler()
scalerNchSmeared = StandardScaler()

scalerNchCutTrue = StandardScaler()
scalerNchCutSmeared = StandardScaler()

# scale the arrays
dataTrainingNchTrueScaled = scalerNchTrue.fit_transform(dataTrainingNchTrue)
dataTrainingNchSmearedScaled = scalerNchSmeared.fit_transform(dataTrainingNchSmeared) # fit on the training data and also scale (transform) them
dataUnfoldingNchSmearedScaled = scalerNchSmeared.transform(dataUnfoldingNchSmeared) # only scale the pseudo-experimental data

dataTrainingNchCutTrueScaled = scalerNchCutTrue.fit_transform(dataTrainingNchCutTrue)
dataTrainingNchCutSmearedScaled = scalerNchCutSmeared.fit_transform(dataTrainingNchCutSmeared) # fit on the training data and also scale (transform) them
dataUnfoldingNchCutSmearedScaled = scalerNchCutSmeared.transform(dataUnfoldingNchCutSmeared) # only scale the pseudo-experimental data

## Perform Unfolding

We perform unfolding of multiplicity via Unifold and of spherocities via MultiFold. We utilize the `omnifold(training true observables, training smeared observables, smeared observables to be unfolded, nIter, network)` function from the `omnifold.py` file.

In [ ]:
# set the number of iterations
nIter = 10

# UniFold
weightsNch, modelHistoryStep1Nch = of.omnifold(dataTrainingNchTrueScaled, dataTrainingNchSmearedScaled, dataUnfoldingNchSmearedScaled, nIter, UniFold)

# MultiFold
weightsNchCut, modelHistoryStep1NchCut = of.omnifold(dataTrainingNchCutTrueScaled, dataTrainingNchCutSmearedScaled, dataUnfoldingNchCutSmearedScaled, nIter, MultiFold)

## Plot the Results

We create a folder for ML unfolding plots. 

In [ ]:
outDir = f"img/OmniFold_{seedTrainingBiased}+{seedTrainingMB}-{seedUnfolding}"
os.makedirs(outDir, exist_ok=True)

We define a Python helper function which fills in a NumPy histogram and also calculates proper statistical weights for each bin.

In [ ]:
def countsAndErrors(data, bins, weights=None, density=True):
    counts, _ = np.histogram(data, bins=bins, weights=weights) # fills in bins based on the data array
    
    # unweighted data (Poisson error)
    if weights is None:
        errors = np.sqrt(counts) # calculates the Poisson errors for each bin

    # weighted ML data (sum of weights squared)
    else:
        sumw2, _ = np.histogram(data, bins=bins, weights=weights**2) # fills in bins with weights squared based on the data array
        errors = np.sqrt(sumw2) # calculates the square root of the sum of weights squared for each bin

    if density:
        binWidths = np.diff(bins) # widths of bins
        integral = np.sum(counts * binWidths)
        if integral!=0:
            counts = counts / integral
            errors = errors / integral
        
    return counts, errors

Next, we prepare binning for the final plots.

In [ ]:
binsNch = np.linspace(0, 80, 81) # the third argument represents the number of bin edges = number of bins + 1
centresNch = 0.5 * (binsNch[:-1] + binsNch[1:]) # calculate the bin centres for closure tests as the average of the bin edges

binsSpherocity = np.linspace(0, 1, 51)
centresSpherocity = 0.5 * (binsSpherocity[:-1] + binsSpherocity[1:])

We show and save technical plots associated with the neural network output. Particularly, we show the training and validation loss, output probabilities which serve as weights, and histogram of weights for the last iteration.

In [ ]:
# loop over neural networks (first UniFold, then MultiFold)
for j, _ in enumerate([(r'$N_\mathrm{ch}$', dataTrainingNchSmeared[:, 0], dataTrainingNchSmearedScaled, weightsNch[-1, 1, :], modelHistoryStep1Nch), ('Spherocity', dataTrainingNchCutSmeared[:, 1], dataTrainingNchCutSmearedScaled, weightsNchCut[-1, 1, :], modelHistoryStep1NchCut)]):
    fig, ax = plt.subplots(1, 3, figsize=(24, 7))
    
    latex, dataObservable, dataObservableScaled, weights, modelHistory = _

    # ----
    # LOSS
    # ----

    ax[0].plot(modelHistory.history['loss'], label='training loss')
    ax[0].plot(modelHistory.history['val_loss'], label='validation loss')

    ax[0].set_title('Loss for the Last Iteration')
    ax[0].set_xlabel('Epochs')
    ax[0].set_ylabel('Loss')

    ax[0].legend()

    # ----------
    # PREDICTION
    # ----------

    prediction = modelHistory.model.predict(dataObservableScaled, batch_size=10000, verbose=0)

    ax[1].scatter(dataObservable, prediction, alpha=0.01, s=1)

    ax[1].set_title('Model Prediction for the Last Iteration')
    ax[1].set_xlabel(latex)
    ax[1].set_ylabel('Predicted Probability')

    # --------------------
    # HISTOGRAM OF WEIGHTS
    # --------------------

    ax[2].hist(weights, alpha=0.5)

    ax[2].set_title('Weights')
    ax[2].set_xlabel('Weights')
    ax[2].set_ylabel('Counts')

    if j==0:
        fig.suptitle(f"Multiplicity UniFold", fontsize=30)
        plt.savefig(f"img/OmniFold_{seedTrainingBiased}+{seedTrainingMB}-{seedUnfolding}/unifold_history.pdf", bbox_inches="tight")
    else:
        fig.suptitle(f"Spherocity MultiFold", fontsize=30)
        plt.savefig(f"img/OmniFold_{seedTrainingBiased}+{seedTrainingMB}-{seedUnfolding}/multifold_history.pdf", bbox_inches="tight")

Ultimately, we plot the unfolding results. 

In [ ]:
# loop over iterations
for iIter in range(nIter):
    
    # define the subplot matrix
    fig, ax = plt.subplots(3, 3, figsize=(24, 21))


    # ===================
    # Row 0: MULTIPLICITY
    # ===================

    # fill histograms and calculate bin errors
    centresTrainingNchTrue, errorsTrainingNchTrue = countsAndErrors(dataTrainingNchTrue[:, 0], binsNch)
    centresTrainingNchSmeared, errorsTrainingNchSmeared = countsAndErrors(dataTrainingNchSmeared[:, 0], binsNch)
    centresUnfoldingNchTrue, errorsUnfoldingNchTrue = countsAndErrors(dataUnfoldingNchTrue[:, 0], binsNch)
    centresUnfoldingNchSmeared, errorsUnfoldingNchSmeared = countsAndErrors(dataUnfoldingNchSmeared[:, 0], binsNch)
    centresUnfoldedNchTrue, errorsUnfoldedNchTrue = countsAndErrors(dataTrainingNchTrue[:, 0], binsNch, weights=weightsNch[iIter, 1, :])
    centresReweightedNchSmeared, errorsReweightedNchSmeared = countsAndErrors(dataTrainingNchSmeared[:, 0], binsNch, weights=weightsNch[iIter, 0, :])

    # ----------------------
    # STEP 1: detector level
    # ----------------------

    # draw histograms as probability densities (density=True) with errorbars
    ax[0][0].hist(dataTrainingNchSmeared[:, 0], bins=binsNch, density=True, color='blue', histtype='step', label='training, smeared')
    ax[0][0].errorbar(centresNch, centresTrainingNchSmeared, yerr=errorsTrainingNchSmeared, fmt='none', ecolor='blue')

    ax[0][0].hist(dataUnfoldingNchSmeared[:, 0], bins=binsNch, density=True, color='green', label='unfolding, smeared', alpha=0.3)
    ax[0][0].errorbar(centresNch, centresUnfoldingNchSmeared, yerr=errorsUnfoldingNchSmeared, fmt='none', ecolor='green', alpha=0.3)

    ax[0][0].hist(dataTrainingNchSmeared[:, 0], weights=weightsNch[iIter, 0, :], bins=binsNch, density=True, color='red', histtype='step', label='reweighted')
    ax[0][0].errorbar(centresNch, centresReweightedNchSmeared, yerr=errorsReweightedNchSmeared, fmt='none', ecolor='red')

    # customize the plot
    ax[0][0].set_title("Step 1: detector level", fontsize=25)
    ax[0][0].set_xlabel(r"$N_\mathrm{ch}$ $[-]$", fontsize=15, loc='right')
    ax[0][0].set_ylabel("Normalized Events", fontsize=15, loc='top')
    ax[0][0].set_yscale('log')

    maxValNchStep1 = max(np.max(centresTrainingNchSmeared), np.max(centresUnfoldingNchSmeared), np.max(centresReweightedNchSmeared))
    ax[0][0].set_ylim(top=maxValNchStep1*5)

    ax[0][0].legend(frameon=False, fontsize=10, loc='upper center')

    # description
    xCoord = 0.67
    yCoord = 0.95

    text_kwargs = { # define dictionary containing common arguments for text labels
        'transform': ax[0][0].transAxes, # use normalized coordinates
        'fontsize': 14, 
        'horizontalalignment': 'left', 
        'verticalalignment': 'center'  
    }

    ax[0][0].text(xCoord, yCoord, r"pp, $\sqrt{s}$ = 510 GeV", **text_kwargs)
    ax[0][0].text(xCoord, yCoord - 0.06, r"MB, PYTHIA8 (Detroit)", **text_kwargs)
    ax[0][0].text(xCoord, yCoord - 0.12, r"$|\eta| < 1$, $p_\mathrm{T} > 0.2$ GeV/c", **text_kwargs)
    ax[0][0].text(xCoord, yCoord - 0.18, "THIS THESIS", weight='bold', **text_kwargs)

    # ----------------------
    # STEP 2: particle level
    # ----------------------

    # draw histograms as probability densities (density=True) with errorbars
    ax[0][1].hist(dataTrainingNchTrue[:, 0], bins = binsNch, density=True, color='blue', histtype='step', label='training, true')
    ax[0][1].errorbar(centresNch, centresTrainingNchTrue, yerr=errorsTrainingNchTrue, fmt='none', ecolor='blue')

    ax[0][1].hist(dataUnfoldingNchTrue[:, 0], bins = binsNch, density=True, color='green', label='unfolding, true', alpha=0.3)
    ax[0][1].errorbar(centresNch, centresUnfoldingNchTrue, yerr=errorsUnfoldingNchTrue, fmt='none', ecolor='green', alpha=0.3)

    ax[0][1].hist(dataTrainingNchTrue[:, 0], weights=weightsNch[iIter, 1, :], bins = binsNch, density=True, color='red', histtype='step', label='unfolded')
    ax[0][1].errorbar(centresNch, centresUnfoldedNchTrue, yerr=errorsUnfoldedNchTrue, fmt='none', ecolor='red')

    # customize the plot
    ax[0][1].set_title("Step 2: particle level", fontsize=25)
    ax[0][1].set_xlabel(r"$N_\mathrm{ch}$ $[-]$", fontsize=15, loc='right')
    ax[0][1].set_ylabel("Normalized Events", fontsize=15, loc='top')
    ax[0][1].set_yscale('log')

    maxValNchStep2 = max(np.max(centresTrainingNchTrue), np.max(centresUnfoldingNchTrue), np.max(centresUnfoldedNchTrue))
    ax[0][1].set_ylim(top=maxValNchStep2*5)

    ax[0][1].legend(frameon=False, fontsize=10, loc='upper center')

    # description
    xCoord = 0.67
    yCoord = 0.95

    text_kwargs = { # define dictionary containing common arguments for text labels
        'transform': ax[0][1].transAxes, # use normalized coordinates
        'fontsize': 14, 
        'horizontalalignment': 'left', 
        'verticalalignment': 'center'  
    }

    ax[0][1].text(xCoord, yCoord, r"pp, $\sqrt{s}$ = 510 GeV", **text_kwargs)
    ax[0][1].text(xCoord, yCoord - 0.06, r"MB, PYTHIA8 (Detroit)", **text_kwargs)
    ax[0][1].text(xCoord, yCoord - 0.12, r"$|\eta| < 1$, $p_\mathrm{T} > 0.2$ GeV/c", **text_kwargs)
    ax[0][1].text(xCoord, yCoord - 0.18, "THIS THESIS", weight='bold', **text_kwargs)

    # ------------
    # CLOSURE TEST
    # ------------

    # calculate closure ratios with errors
    ratioNch = np.divide(centresUnfoldedNchTrue, centresUnfoldingNchTrue, out=np.zeros_like(centresUnfoldedNchTrue), where=centresUnfoldingNchTrue!=0) # divide the unfolded values by the true values
    errRatioNch = ratioNch * np.sqrt(
        np.divide(errorsUnfoldedNchTrue, centresUnfoldedNchTrue, out=np.zeros_like(errorsUnfoldedNchTrue), where=centresUnfoldedNchTrue!=0)**2 + 
        np.divide(errorsUnfoldingNchTrue, centresUnfoldingNchTrue, out=np.zeros_like(errorsUnfoldingNchTrue), where=centresUnfoldingNchTrue!=0)**2
    ) # the division error propagation

    # draw the histogram as points with errorbars
    ax[0][2].errorbar(centresNch, ratioNch, yerr=errRatioNch, fmt='o', color='deeppink', ecolor='deeppink', markersize=5)

    # draw the reference line at y=1
    ax[0][2].axhline(1.0, color='green', linestyle='--', zorder=3) 

    # customize the plot
    ax[0][2].set_title("Closure Test", fontsize=25)
    ax[0][2].set_xlabel(r"$N_{\mathrm{ch}}$ $[-]$", fontsize=15, loc='right')
    ax[0][2].set_ylabel("Unfolded / True", fontsize=15, loc='top')
    ax[0][2].set_ylim(0.7, 1.3) 

    # description
    xCoord = 0.37
    yCoord = 0.24

    text_kwargs = { # define dictionary containing common arguments for text labels
        'transform': ax[0][2].transAxes, # use normalized coordinates
        'fontsize': 14, 
        'horizontalalignment': 'left', 
        'verticalalignment': 'center'  
    }

    ax[0][2].text(xCoord, yCoord, r"pp, $\sqrt{s}$ = 510 GeV", **text_kwargs)
    ax[0][2].text(xCoord, yCoord - 0.06, r"MB, PYTHIA8 (Detroit)", **text_kwargs)
    ax[0][2].text(xCoord, yCoord - 0.12, r"$|\eta| < 1$, $p_\mathrm{T} > 0.2$ GeV/c", **text_kwargs)
    ax[0][2].text(xCoord, yCoord - 0.18, "THIS THESIS", weight='bold', **text_kwargs)


    # =======================
    # Rows 1, 2: SPHEROCITIES
    # =======================

    abbrevs = ["S0", "S0pT1"]
    latexs = [r'$S_{0}^{}$', r'$S_{0}^{p_\mathrm{T} = 1}$']

    for j, labels in enumerate(zip(abbrevs, latexs), 1):
        abbrev, latex = labels

        # fill histograms and calculate bin errors
        centresTrainingNchCutTrue, errorsTrainingNchCutTrue = countsAndErrors(dataTrainingNchCutTrue[:, j], binsSpherocity)
        centresTrainingNchCutSmeared, errorsTrainingNchCutSmeared = countsAndErrors(dataTrainingNchCutSmeared[:, j], binsSpherocity)
        centresUnfoldingNchCutTrue, errorsUnfoldingNchCutTrue = countsAndErrors(dataUnfoldingNchCutTrue[:, j], binsSpherocity)
        centresUnfoldingNchCutSmeared, errorsUnfoldingNchCutSmeared = countsAndErrors(dataUnfoldingNchCutSmeared[:, j], binsSpherocity)
        centresUnfoldedNchCutTrue, errorsUnfoldedNchCutTrue = countsAndErrors(dataTrainingNchCutTrue[:, j], binsSpherocity, weights=weightsNchCut[iIter, 1, :])
        centresReweightedNchCutSmeared, errorsReweightedNchCutSmeared = countsAndErrors(dataTrainingNchCutSmeared[:, j], binsSpherocity, weights=weightsNchCut[iIter, 0, :])

        # ----------------------
        # STEP 1: detector level
        # ----------------------

        # draw histograms as probability densities (density=True) with errorbars
        ax[j][0].hist(dataTrainingNchCutSmeared[:, j], bins=binsSpherocity, density=True, color='blue', histtype='step', label='training, smeared')
        ax[j][0].errorbar(centresSpherocity, centresTrainingNchCutSmeared, yerr=errorsTrainingNchCutSmeared, fmt='none', ecolor='blue')

        ax[j][0].hist(dataUnfoldingNchCutSmeared[:, j], bins=binsSpherocity, density=True, color='green', label='unfolding, smeared', alpha=0.3)
        ax[j][0].errorbar(centresSpherocity, centresUnfoldingNchCutSmeared, yerr=errorsUnfoldingNchCutSmeared, fmt='none', ecolor='green', alpha=0.3)

        ax[j][0].hist(dataTrainingNchCutSmeared[:, j], weights=weightsNchCut[iIter, 0, :], bins=binsSpherocity, density=True, color='red', histtype='step', label='reweighted')
        ax[j][0].errorbar(centresSpherocity, centresReweightedNchCutSmeared, yerr=errorsReweightedNchCutSmeared, fmt='none', ecolor='red')

        # customize the plot
        ax[j][0].set_xlabel(latex+" $[-]$", fontsize=15, loc='right')
        ax[j][0].set_ylabel("Normalized Events", fontsize=15, loc='top')
        ax[j][0].set_yscale('log')

        maxValNchCutStep1 = max(np.max(centresTrainingNchCutSmeared), np.max(centresUnfoldingNchCutSmeared), np.max(centresReweightedNchCutSmeared))
        ax[j][0].set_ylim(top=maxValNchCutStep1*7)

        ax[j][0].legend(frameon=False, fontsize=10, loc='upper center')

        # description
        xCoord = 0.37
        yCoord = 0.3

        text_kwargs = { # define dictionary containing common arguments for text labels
            'transform': ax[j][0].transAxes, # use normalized coordinates
            'fontsize': 14, 
            'horizontalalignment': 'left', 
            'verticalalignment': 'center'  
        }

        ax[j][0].text(xCoord, yCoord, r"pp, $\sqrt{s}$ = 510 GeV", **text_kwargs)
        ax[j][0].text(xCoord, yCoord - 0.06, r"MB, PYTHIA8 (Detroit)", **text_kwargs)
        ax[j][0].text(xCoord, yCoord - 0.12, r"$|\eta| < 1$, $p_\mathrm{T} > 0.2$ GeV/c", **text_kwargs)
        ax[j][0].text(xCoord, yCoord - 0.18, r"$N_\mathrm{ch}^\mathrm{MC} > 10$, $N_\mathrm{ch}^\mathrm{sm} > 10$", **text_kwargs)
        ax[j][0].text(xCoord, yCoord - 0.24, "THIS THESIS", weight='bold', **text_kwargs)

        # ----------------------
        # STEP 2: particle level
        # ----------------------

        # draw histograms with errorbars
        ax[j][1].hist(dataTrainingNchCutTrue[:, j], bins=binsSpherocity, density=True, color='blue', histtype='step', label='training, true')
        ax[j][1].errorbar(centresSpherocity, centresTrainingNchCutTrue, yerr=errorsTrainingNchCutSmeared, fmt='none', ecolor='blue')

        ax[j][1].hist(dataUnfoldingNchCutTrue[:, j], bins=binsSpherocity, density=True, color='green', label='unfolding, true', alpha=0.3)
        ax[j][1].errorbar(centresSpherocity, centresUnfoldingNchCutTrue, yerr=errorsUnfoldingNchCutTrue, fmt='none', ecolor='green', alpha=0.3)

        ax[j][1].hist(dataTrainingNchCutTrue[:, j], weights=weightsNchCut[iIter, 1, :], bins=binsSpherocity, density=True, color='red', histtype='step', label='unfolded')
        ax[j][1].errorbar(centresSpherocity, centresUnfoldedNchCutTrue, yerr=errorsUnfoldedNchCutTrue, fmt='none', ecolor='red')

        # customize the plot
        ax[j][1].set_xlabel(latex+" $[-]$", fontsize=15, loc='right')
        ax[j][1].set_ylabel("Normalized Events", fontsize=15, loc='top')
        ax[j][1].set_yscale('log')

        maxValNchCutStep2 = max(np.max(centresTrainingNchCutTrue), np.max(centresUnfoldingNchCutTrue), np.max(centresUnfoldedNchCutTrue))
        ax[j][1].set_ylim(top=maxValNchCutStep2*7)

        ax[j][1].legend(frameon=False, fontsize=10, loc='upper center')

        # description
        xCoord = 0.37
        yCoord = 0.3

        text_kwargs = { # define dictionary containing common arguments for text labels
            'transform': ax[j][1].transAxes, # use normalized coordinates
            'fontsize': 14, 
            'horizontalalignment': 'left', 
            'verticalalignment': 'center'  
        }

        ax[j][1].text(xCoord, yCoord, r"pp, $\sqrt{s}$ = 510 GeV", **text_kwargs)
        ax[j][1].text(xCoord, yCoord - 0.06, r"MB, PYTHIA8 (Detroit)", **text_kwargs)
        ax[j][1].text(xCoord, yCoord - 0.12, r"$|\eta| < 1$, $p_\mathrm{T} > 0.2$ GeV/c", **text_kwargs)
        ax[j][1].text(xCoord, yCoord - 0.18, r"$N_\mathrm{ch}^\mathrm{MC} > 10$, $N_\mathrm{ch}^\mathrm{sm} > 10$", **text_kwargs)
        ax[j][1].text(xCoord, yCoord - 0.24, "THIS THESIS", weight='bold', **text_kwargs)

        # ------------
        # CLOSURE TEST
        # ------------

        # calculate closure ratios with errors
        ratioNchCut = np.divide(centresUnfoldedNchCutTrue, centresUnfoldingNchCutTrue, out=np.zeros_like(centresUnfoldedNchCutTrue), where=centresUnfoldingNchCutTrue!=0) # divide the unfolded values by the true values
        errRatioNchCut = ratioNchCut * np.sqrt(
            np.divide(errorsUnfoldedNchCutTrue, centresUnfoldedNchCutTrue, out=np.zeros_like(errorsUnfoldedNchCutTrue), where=centresUnfoldedNchCutTrue!=0)**2 + 
            np.divide(errorsUnfoldingNchCutTrue, centresUnfoldingNchCutTrue, out=np.zeros_like(errorsUnfoldingNchCutTrue), where=centresUnfoldingNchCutTrue!=0)**2
        ) # the division error propagation

        # draw the histogram as points with errorbars
        ax[j][2].errorbar(centresSpherocity, ratioNchCut, yerr=errRatioNchCut, fmt='o', color='deeppink', ecolor='deeppink', markersize=5)

        # draw the reference line at y=1
        ax[j][2].axhline(1.0, color='green', linestyle='--', zorder=3)

        # customize the plot
        ax[j][2].set_xlabel(latex+" $[-]$", fontsize=15, loc='right')
        ax[j][2].set_ylabel("Unfolded / True", fontsize=15, loc='top')
        ax[j][2].set_ylim(0.7, 1.3) 

        # description
        xCoord = 0.37
        yCoord = 0.3

        text_kwargs = { # define dictionary containing common arguments for text labels
            'transform': ax[j][2].transAxes, # use normalized coordinates
            'fontsize': 14, 
            'horizontalalignment': 'left', 
            'verticalalignment': 'center'  
        }

        ax[j][2].text(xCoord, yCoord, r"pp, $\sqrt{s}$ = 510 GeV", **text_kwargs)
        ax[j][2].text(xCoord, yCoord - 0.06, r"MB, PYTHIA8 (Detroit)", **text_kwargs)
        ax[j][2].text(xCoord, yCoord - 0.12, r"$|\eta| < 1$, $p_\mathrm{T} > 0.2$ GeV/c", **text_kwargs)
        ax[j][2].text(xCoord, yCoord - 0.18, r"$N_\mathrm{ch}^\mathrm{MC} > 10$, $N_\mathrm{ch}^\mathrm{sm} > 10$", **text_kwargs)
        ax[j][2].text(xCoord, yCoord - 0.24, "THIS THESIS", weight='bold', **text_kwargs)

    # draw iteration number
    fig.suptitle(f"Iteration {iIter}", fontsize=30)

    # save
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"img/OmniFold_{seedTrainingBiased}+{seedTrainingMB}-{seedUnfolding}/iteration{iIter}.pdf", bbox_inches="tight")